## Knihovny

In [2]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Úloha 1

## Lineární oscilátor

In [ ]:
# Lineární oscilátor - těleso na pružině
def linearni_oscilator(t, y, delta, alpha=1.0):
    x, v = y
    return [v, -delta * v - alpha * x]

t_span = (0, 100)
t_eval = np.linspace(0, 50, 1000)
poc_podminky = [2.0, 0.0] # Počáteční výchylka 2, nulová rychlost

# Režimy
res_slabe = solve_ivp(linearni_oscilator, t_span, poc_podminky, args=(0.25,), t_eval=t_eval)
res_krit = solve_ivp(linearni_oscilator, t_span, poc_podminky, args=(2.0,), t_eval=t_eval)
res_silne = solve_ivp(linearni_oscilator, t_span, poc_podminky, args=(4.0,), t_eval=t_eval)

# Vykreslování
fig_lin = go.Figure()
fig_lin.add_trace(go.Scatter(x=res_slabe.t, y=res_slabe.y[0], name="Slabé (delta=0.5)"))
fig_lin.add_trace(go.Scatter(x=res_krit.t, y=res_krit.y[0], name="Kritické (delta=2.0)"))
fig_lin.add_trace(go.Scatter(x=res_silne.t, y=res_silne.y[0], name="Silné (delta=4.0)"))
fig_lin.update_layout(title="Lineární oscilátor: Časový průběh x(t)", template="plotly_dark")
fig_lin.show()


# Fázový diagram
fig_lin_faz = go.Figure()

# res_slabe.y[0] je poloha (x), res_slabe.y[1] je rychlost (v)
fig_lin_faz.add_trace(go.Scatter(x=res_slabe.y[0], y=res_slabe.y[1], name="Slabé (delta=0.25)"))
fig_lin_faz.add_trace(go.Scatter(x=res_krit.y[0], y=res_krit.y[1], name="Kritické (delta=2.0)"))
fig_lin_faz.add_trace(go.Scatter(x=res_silne.y[0], y=res_silne.y[1], name="Silné (delta=4.0)"))

fig_lin_faz.update_layout(
    title="Lineární oscilátor: Fázový diagram (x vs v)",
    xaxis_title="Poloha (x)",
    yaxis_title="Rychlost (v)",
    template="plotly_dark"
)

fig_lin_faz.show()


### Otázka: Co odlišuje slabé, kritické a silné tlumení?
**Slabé tlumení:** Odpor prostředí je malý. Těleso překmitne přes rovnovážnou polohu a postupně se s klesající amplitudou zhoupne do klidu.

**Kritické tlumení:** Hraniční stav. Těleso se vrátí do rovnovážné polohy v nejkratším možném čase, aniž by ji překmitlo.

**Silné tlumení:** Odpor je velmi silný. Těleso rovnováhu nepřekmitne, ale návrat do nuly trvá mnohem déle než u kritického tlumení

### Otázka: Jaký mechanismus vede k rezonanci u buzeného kmitání?

Stav, při kterém se frekvence vnější budící síly přesně shoduje s přirozenou (vlastní) frekvencí oscilátoru. Př.: Představme si houpání na houpačce – pokud do ní strkáme přesně v rytmu jejího přirozeného zhoupnutí, energie se s každým strčením kumuluje a amplituda (rozsah) kmitů dramaticky roste.

### Otázka: Dá se problém řešit pomocí symbolické matematiky? Jak?

Záleží na typu oscilátoru. Lineární oscilátor (těleso na pružině) lze snadno vyřešit symbolicky (např. pomocí knihovny sympy v Pythonu nebo na papíře). Výsledkem je přesný matematický vzorec (obsahující goniometrické funkce a exponenciálu). Nelineární Duffingův oscilátor však vykazuje chaotické chování a analytické (symbolické) řešení pro něj neexistuje. Musí se řešit výhradně numericky (krok za krokem), jak to dělá funkce solve_ivp.

## Duffingův oscilátor

In [ ]:
def duffing(t, y, delta, alpha, beta, gamma, omega):
    x, v = y
    return [v, gamma * np.cos(omega * t) - delta * v - alpha * x - beta * (x**3)]

# Parametry ze zadání - tvar misky, síla buzení
delta, alpha, beta, gamma, omega = 0.2, -1.0, 1.0, 0.3, 1.2
t_span = (0, 500)
t_eval = np.linspace(0, 500, 10000)

# Náhodná počáteční poloha i rychlost - [-2, 2]
x0 = np.random.uniform(-2.0, 2.0)
v0 = np.random.uniform(-2.0, 2.0)

print(f"Vygenerován náhodný start: Poloha (x0) = {x0:.3f}, Rychlost (v0) = {v0:.3f}")

# Výpočet spojité křivky
res_duffing = solve_ivp(duffing, t_span, [x0, v0],
                        args=(delta, alpha, beta, gamma, omega), t_eval=t_eval)

# Výpočet Poincarého průřezu
T = 2 * np.pi / omega
t_poincare = np.arange(0, t_span[1], T)
res_poincare = solve_ivp(duffing, t_span, [x0, v0],
                         args=(delta, alpha, beta, gamma, omega), t_eval=t_poincare)

# Vykreslení grafu
fig = go.Figure()

# Přidání dráhy
fig.add_trace(go.Scatter(x=res_duffing.y[0], y=res_duffing.y[1],
                         mode="lines", name="Fázový diagram", opacity=0.3))

# Přidání červených bodů
fig.add_trace(go.Scatter(x=res_poincare.y[0], y=res_poincare.y[1],
                         mode="markers", name="Poincarého průřez",
                         marker=dict(size=5, color="red")))
# Úprava vzhledu
fig.update_layout(title=f"Duffingův oscilátor (Start: x={x0:.2f}, v={v0:.2f})",
                  xaxis_title="Poloha (x)", yaxis_title="Rychlost (v)",
                  template="plotly_dark",
                  width=800, height=600)

fig.show()

Vygenerován náhodný start: Poloha (x0) = -0.506, Rychlost (v0) = 1.764


### Otázka: Dochází pro dané parametry k chaotickému chování?
Ano. Důkazem je Poincarého průřez (červené body v grafu). Kdyby byl pohyb periodický a pravidelný, stroboskop by zaznamenal jen jeden nebo několik málo izolovaných bodů. Místo toho body vykreslují složitý fraktální útvar, což je typický znak deterministického chaosu.
### Otázka: Jaký je rozdíl mezi malou a velkou počáteční podmínkou?
Malá počáteční podmínka (např. kulička puštěná blízko dna jedné z jam) způsobí, že těleso může zpočátku chvíli kmitat jen v jedné jámě, než nasbírá energii k přeskočení. Velká počáteční podmínka (velká výchylka nebo rychlost) způsobí okamžité překonávání středového hrbolku. Zásadní ale je, že díky podivnému atraktoru obě dráhy po čase „zapomenou“ svůj start a nakonec se usadí na tom samém chaotickém obrazci.
### Otázka: Co se stane, když zvýšíte buzení $\gamma$?
Parametr $\gamma$ určuje sílu, kterou do misky zvenčí "strkáme". Zvýšení této síly dodá systému více energie. To vede k takzvaným bifurkacím – systém může skokově přejít z plného chaosu do pravidelného periodického pohybu a při dalším zvyšování zase zpět do chaosu.
### Otázka: Mění se počet atraktorů?
Ano, počet atraktorů silně závisí na parametrech. Pro slabé buzení mohou existovat dva samostatné bodové atraktory (kulička prostě spadne do levé nebo pravé jámy). Při zadaných parametrech existuje jeden velký chaotický (podivný) atraktor, který pokrývá obě jámy.

## Heatmapa amplitudy

In [ ]:
def duffing(t, y, delta, alpha, beta, gamma, omega):
    x, v = y
    return [v, gamma * np.cos(omega * t) - delta * v - alpha * x - beta * (x**3)]

# Pevné parametry
delta = 0.2
alpha = -1.0
beta = 1.0

# Mřížka 20 * 20 => 400 simulací

# Tvorba hodnot 20 pro každou
gamma_hodnoty = np.linspace(0.1, 0.6, 20)
omega_hodnoty = np.linspace(0.5, 1.5, 20)

# Prázdná matice, pro výsledné amplitudy
amplitudy = np.zeros((len(gamma_hodnoty), len(omega_hodnoty)))

# Čas pro simulace
t_span = (0, 100)
t_eval = np.linspace(0, 100, 1000)

print("Heatmapa se počítá (400 simulací), zabere to pár vteřin...")

# Smyčky pro výpočet
for i, gamma in enumerate(gamma_hodnoty):
    for j, omega in enumerate(omega_hodnoty):

        # Simulace pro danou dvojici (gamma, omega)
        res = solve_ivp(duffing, t_span, [1.0, 0.0],
                        args=(delta, alpha, beta, gamma, omega), t_eval=t_eval)

        # Oříznutí hodnot od bodu 500. pro zachycení pouze ustáleného stavu
        x_ustalene = res.y[0][500:]

        # Výpočet amplitudy -> Nejvyšší bod - Nejnižší bod
        amplituda = np.max(x_ustalene) - np.min(x_ustalene)

        amplitudy[i, j] = amplituda


# Vykreslení grafu heatmapy
fig = go.Figure(data=go.Heatmap(
    z=amplitudy,
    x=omega_hodnoty,
    y=gamma_hodnoty,
    colorscale='Inferno'
))

fig.update_layout(
    title="Heatmapa amplitudy Duffingova oscilátoru",
    xaxis_title="Frekvence buzení (ω) - Rytmus",
    yaxis_title="Síla buzení (γ) - Výkon",
    template="plotly_dark",
    width=700, height=600
)

fig.show()

Počítám heatmapu (400 simulací), vyčkej pár vteřin...
Hotovo, vykresluji!


Tento graf zobrazuje reakci Duffingova oscilátoru na 400 různých kombinací budící síly ($\gamma$) a frekvence ($\omega$). Abychom měřili skutečnou odezvu systému, algoritmus ignoruje první polovinu naměřených dat (transientní jev) a zaznamenává pouze amplitudu v ustáleném stavu. Světlé oblasti představují rezonanci a chaotické chování – systém pohlcuje maximum energie z vnějšího buzení a kulička s velkým rozkmitem přeskakuje mezi oběma jámami. Naopak tmavé oblasti ukazují kombinace parametrů, kdy je kulička uvězněna v jedné jámě a kmitá jen s minimální amplitudou, protože vnější síla s přirozeným chováním systému "neladí".

# Úloha 2

## SIR model

In [4]:
def sir_rovnice(t, y, N, beta, gamma):
    S, I, R = y
    return [-beta * S * I / N, (beta * S * I / N) - gamma * I, gamma * I]

N = 1000000 # Populace 1 milion
trvani_simulace = 400
t_eval = np.linspace(0, trvani_simulace, trvani_simulace)

# 1. Tuberkulóza
vysl_tbc = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.26, 0.1), t_eval=t_eval)

# 2. Chřipka
vysl_chripka = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.35, 0.1), t_eval=t_eval)

# 3. Spalničky
vysl_spal = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 1.70, 0.1), t_eval=t_eval)

# 4. Pravé neštovice
vysl_prn = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.50, 0.1), t_eval=t_eval)

# 5. Plané neštovice
vysl_pln = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 1.00, 0.1), t_eval=t_eval)

# Nemoci mimo tabulku

# 1. Španělská chřipka
vysl_span = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.20, 0.1), t_eval=t_eval)

# 2. COVID-19
vysl_covid = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.28, 0.1), t_eval=t_eval)

# 3. Rýma
vysl_ryma = solve_ivp(sir_rovnice, (0, trvani_simulace), [N-1, 1, 0], args=(N, 0.15, 0.1), t_eval=t_eval)


# Tvorba grafu
fig = go.Figure()

# Křivka: Tuberkulóza
fig.add_trace(go.Scatter(x=vysl_tbc.t, y=vysl_tbc.y[1], name="Tuberkulóza", visible=True))

# Křivka: Chřipka
fig.add_trace(go.Scatter(x=vysl_chripka.t, y=vysl_chripka.y[1], name="Chřipka", visible=False))

# Křivka: Spalničky
fig.add_trace(go.Scatter(x=vysl_spal.t, y=vysl_spal.y[1], name="Spalničky", visible=False))

# Křivka: Pravé neštovice
fig.add_trace(go.Scatter(x=vysl_prn.t, y=vysl_prn.y[1], name="Pravé neštovice", visible=False))

# Křivka: Plané neštovice
fig.add_trace(go.Scatter(x=vysl_pln.t, y=vysl_pln.y[1], name="Plané neštovice", visible=False))

# Křivka: Rýma
fig.add_trace(go.Scatter(x=vysl_ryma.t, y=vysl_ryma.y[1], name="Rýma", visible=False))

# Křivka: Španělská chřipka
fig.add_trace(go.Scatter(x=vysl_span.t, y=vysl_span.y[1], name="Španělská chřipka", visible=True))

# Křivka: COVID-19
fig.add_trace(go.Scatter(x=vysl_covid.t, y=vysl_covid.y[1], name="COVID-19", visible=False))

# Posuvník pro nemoci
#TODO - přidat posuvník pro dny

kroky = [
    dict(method="update", args=[{"visible": [True, False, False, False, False, False, False, False]}], label="Tuberkulóza"),
    dict(method="update", args=[{"visible": [False, True, False, False, False, False, False, False]}], label="Chřipka"),
    dict(method="update", args=[{"visible": [True, True, False, False, False, False, False, False]}], label="Porovnání TBC a Chřipky"),
    dict(method="update", args=[{"visible": [False, False, True, False, False, False, False, False]}], label="Spalničky"),
    dict(method="update", args=[{"visible": [False, False, False, True, False, False, False, False]}], label="Pravé neštovice"),
    dict(method="update", args=[{"visible": [False, False, False, False, True, False, False, False]}], label="Plané neštovice"),
    dict(method="update", args=[{"visible": [False, False, False, True, True, False, False, False]}], label="Porovnání Pravé, Plané neštovice"),
    dict(method="update", args=[{"visible": [True, True, True, True, True, False, False, False]}], label="Porovnání všech uvedených"),

    dict(method="update", args=[{"visible": [False, False, False, False, False, True, False, False]}], label="Rýma"),
    dict(method="update", args=[{"visible": [False, False, False, False, False, False, True, False]}], label="Španělská chřipka"),
    dict(method="update", args=[{"visible": [False, False, False, False, False, False, False, True]}], label="COVID-19"),
    dict(method="update", args=[{"visible": [False, False, False, False, False, False, True, True]}], label="Porovnání Covid_19, Španělská chřipka"),
    dict(method="update", args=[{"visible": [True, True, True, True, True, True, True, True]}], label="Porovnání všech uvedených")
]

# Přidání posuvníku a stylizace
fig.update_layout(
    sliders=[dict(active=0, steps=kroky)],
    template="plotly_dark",
    title="Průběh infekce: Počet aktuálně nakažených (I)",
    xaxis_title="Dny od začátku epidemie",
    yaxis_title="Počet nakažených lidí"
)

fig.show()


print(f"\n--- CELKOVÉ STATISTIKY PO {trvani_simulace} DNECH ---")

# Tuberkulóza
zdravi_tbc = int(vysl_tbc.y[0][-1])
print(f"Tuberkulóza: Onemocnělo {N - zdravi_tbc:,} lidí. Neonemocnělo {zdravi_tbc:,} lidí.")

# Chřipka
zdravi_chripka = int(vysl_chripka.y[0][-1])
print(f"Chřipka: Onemocnělo {N - zdravi_chripka:,} lidí. Neonemocnělo {zdravi_chripka:,} lidí.")

# Spalničky
zdravi_spal = int(vysl_spal.y[0][-1])
print(f"Spalničky: Onemocnělo {N - zdravi_spal:,} lidí. Neonemocnělo {zdravi_spal:,} lidí.")

# Pravé neštovice
zdravi_prn = int(vysl_prn.y[0][-1])
print(f"Pravé neštovice: Onemocnělo {N - zdravi_prn:,} lidí. Neonemocnělo {zdravi_prn:,} lidí.")

# Plané neštovice
zdravi_pln = int(vysl_pln.y[0][-1])
print(f"Plané neštovice: Onemocnělo {N - zdravi_pln:,} lidí. Neonemocnělo {zdravi_pln:,} lidí.")

# Rýma
zdravi_ryma = int(vysl_ryma.y[0][-1])
print(f"Rýma: Onemocnělo {N - zdravi_ryma:,} lidí. Neonemocnělo {zdravi_ryma:,} lidí.")

# Španělská chřipka
zdravi_span = int(vysl_span.y[0][-1])
print(f"Španělská chřipka: Onemocnělo {N - zdravi_span:,} lidí. Neonemocnělo {zdravi_span:,} lidí.")

# COVID-19
zdravi_covid = int(vysl_covid.y[0][-1])
print(f"COVID-19: Onemocnělo {N - zdravi_covid:,} lidí. Neonemocnělo {zdravi_covid:,} lidí.")


--- CELKOVÉ STATISTIKY PO 400 DNECH ---
Tuberkulóza: Onemocnělo 905,437 lidí. Neonemocnělo 94,563 lidí.
Chřipka: Onemocnělo 966,451 lidí. Neonemocnělo 33,549 lidí.
Spalničky: Onemocnělo 1,000,000 lidí. Neonemocnělo 0 lidí.
Pravé neštovice: Onemocnělo 993,102 lidí. Neonemocnělo 6,898 lidí.
Plané neštovice: Onemocnělo 999,954 lidí. Neonemocnělo 46 lidí.
Rýma: Onemocnělo 580,960 lidí. Neonemocnělo 419,040 lidí.
Španělská chřipka: Onemocnělo 798,254 lidí. Neonemocnělo 201,746 lidí.
COVID-19: Onemocnělo 925,502 lidí. Neonemocnělo 74,498 lidí.


Tuberkulóza ($R_0 = 2.6$)  
Vrchol epidemie: Přibližně 47. den.  
Doba trvání: Křivka nákazy klesne k nule zhruba po 130 dnech.  
Konečná bilance: Nákazou projde cca 918 000 lidí.  
Zcela ušetřeno zůstane 82 000 obyvatel.

Chřipka ($R_0 = 3.5$)  
Vrchol epidemie: Přibližně 32. den.  
Doba trvání: Epidemie odezní zhruba za 90 dní.  
Konečná bilance: Onemocní 966 000 obyvatel, nákaze se vyhne 34 000 lidí.  

Pravé neštovice ($R_0 = 5.0$)  
Vrchol epidemie: Přibližně 22. den (velmi strmý nástup).  
Doba trvání: Epidemie vymizí za pouhých 65 dní.  
Konečná bilance: Nakazí se 993 000 lidí (99,3 % populace).  Neonemocní pouze 7 000 jedinců.  

Plané neštovice ($R_0 = 10.0$)  
Vrchol epidemie: Přibližně 11. den.  
Doba trvání: Extrémně rychlé vyhoření vlny během 35 dní.  
Konečná bilance: Onemocní téměř 100 % populace (více než 999 900 lidí). Ušetřeny zůstanou pouze desítky jedinců.  

Spalničky ($R_0 = 17.0$)  
Vrchol epidemie: Extrémní kolaps systému hned 7. den.  
Doba trvání: Není koho dál nakazit, vlna mizí po 25 dnech.  
Konečná bilance: Nakazí se kompletně celá populace (1 000 000 lidí).

Rýma je endemická

Čím více je nemoc nakažlivá, tím dříve dosáhne vrcholu a opět klesne (Rychlejší průběh).

# Úloha 3


## Graf šíření požáru

In [7]:
velikost = 80
pocet_kroku = 60

# Inicializace nedokonalé mřížky
mrizka = np.random.choice([0, 1], size=(velikost, velikost), p=[0.2, 0.8])
mrizka[velikost//2, velikost//2] = 2 # Ohnisko

# Šíření
def krok_pozaru_nahodne(stara_mrizka, sance_na_vzniceni=0.55):
    nova_mrizka = stara_mrizka.copy()
    for i in range(1, velikost - 1):
        for j in range(1, velikost - 1):
            stav = stara_mrizka[i, j]

            if stav == 2:
                nova_mrizka[i, j] = 0
            elif stav == 1:
                okoli = stara_mrizka[i-1:i+2, j-1:j+2]
                if 2 in okoli and np.random.rand() < sance_na_vzniceni:
                    nova_mrizka[i, j] = 2
    return nova_mrizka

# Předpočet všech kroků do paměti
historie_mrizek = [mrizka.copy()]
for _ in range(pocet_kroku):
    mrizka = krok_pozaru_nahodne(mrizka)
    historie_mrizek.append(mrizka.copy())

# Vytvoření základního grafu - krok 0
fig = go.Figure(
    data=[go.Heatmap(
        z=historie_mrizek[0],
        colorscale=[[0, '#2d2d2d'], [0.5, '#228b22'], [1, '#ff4500']],
        zmin=0, zmax=2, showscale=False
    )]
)

# Vytvoření kroků pro posuvník
kroky = []
for i in range(len(historie_mrizek)):
    krok = dict(
        method="restyle",
        args=[{"z": [historie_mrizek[i]]}],
        label=f"Den {i}"
    )
    kroky.append(krok)

# Finální stylování
fig.update_layout(
    sliders=[dict(active=0, steps=kroky)],
    template="plotly_dark",
    title="Interaktivní simulace šíření lesního požáru",
    width=700, height=700,
    yaxis=dict(autorange="reversed") # Převrácení osy Y pro klasický pohled shora
)

fig.show()

Pro třetí úlohu byl vytvořen stochastický mřížkový model (celulární automat) simulující šíření lesního požáru. Místo deterministického pravidla, kde se oheň šíří jako dokonalý geometrický čtverec, zavádí model prvek náhody. Každý zdravý strom sousedící se stromem hořícím má pouze 55% šanci na vzplanutí. Model prokazuje, jak se oheň šíří ve formě nepravidelných fraktálů, a jasně vizualizuje, že pokud plameny narazí na větší shluk nehořlavých polí (kamení, cesty), může se postup ohně zcela zastavit.